# Sensor Repair — Notebook 01: Data preparation and participant splits

**Run on a CPU runtime.** This notebook prepares both datasets for the keep/mask/reconstruct experiments.

1. In Google Drive, create `My Drive/SensorRepair_ICDSIS/raw`.
2. Upload the **two original ZIP downloads** into that folder. Their filenames can stay unchanged.
3. In Colab, choose **Runtime → Run all** and authorize Drive access.

Keep the ZIPs compressed in Drive. The notebook handles nested ZIPs and extracts locally,
avoiding thousands of small Drive reads. It supports the official UCI dataset archives,
rather than CSV-only copies from other sources.

Outputs go to `My Drive/SensorRepair_ICDSIS/processed/v1`.
Share the final displayed summary, `dataset_summary.csv`, and `data_integrity.json` after the run.

The research evaluation will use artificial faults injected into real recordings.
Two-dataset evaluation means the same procedure is trained and evaluated separately on each dataset.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from collections import Counter
import hashlib
import json
import os
import platform
import re
import shutil
import sys
import time
import zipfile
import numpy as np

PROJECT = Path('/content/drive/MyDrive/SensorRepair_ICDSIS')
RAW = PROJECT / 'raw'
OUT = PROJECT / 'processed' / 'v1'
LOCAL = Path('/content/sensorrepair_notebook01')
SEED = 2026

RAW.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
LOCAL.mkdir(parents=True, exist_ok=True)
archives = sorted(p for p in RAW.iterdir() if p.is_file() and p.suffix.lower() == '.zip')
if len(archives) != 2:
    raise ValueError(f'Upload exactly the two original dataset ZIPs into {RAW}. Found {len(archives)} ZIPs.')
for archive in archives:
    print(archive.name, f'{archive.stat().st_size / 1_000_000:.1f} MB')
print('Output folder:', OUT)


Mounted at /content/drive
daily+and+sports+activities.zip 170.8 MB
human+activity+recognition+using+smartphones.zip 61.0 MB
Output folder: /content/drive/MyDrive/SensorRepair_ICDSIS/processed/v1


## Extract the archives

Archive hashes record exactly which downloaded files were used. Extraction is cached locally;
rerunning in the same runtime reuses completed extractions. Originals in Drive are preserved.


In [2]:
def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def extract_zip(source, destination):
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    base = destination.resolve()
    with zipfile.ZipFile(source) as archive:
        for member in archive.infolist():
            target = (base / member.filename).resolve()
            if target != base and base not in target.parents:
                raise ValueError(f'Archive member has an invalid path: {member.filename}')
        archive.extractall(destination)

def unpack_dataset_archive(source, local_root):
    archive_hash = file_sha256(source)
    destination = Path(local_root) / 'extracted' / archive_hash[:16]
    marker = destination / '_extraction_complete.json'
    if marker.exists():
        return destination, archive_hash
    destination.mkdir(parents=True, exist_ok=True)
    local_zip = destination / '_original.zip'
    shutil.copyfile(source, local_zip)
    extract_zip(local_zip, destination)
    processed = {local_zip.resolve()}
    for depth in range(5):
        nested = sorted(p for p in destination.rglob('*')
                        if p.is_file() and p.suffix.lower() == '.zip' and p.resolve() not in processed)
        if not nested:
            break
        for inner in nested:
            processed.add(inner.resolve())
            extract_zip(inner, inner.parent / (inner.stem + '_unpacked'))
    remaining = [p for p in destination.rglob('*')
                 if p.is_file() and p.suffix.lower() == '.zip' and p.resolve() not in processed]
    if remaining:
        raise ValueError('More than five nested archive levels; check the original downloads.')
    marker.write_text(json.dumps({'source_sha256': archive_hash}), encoding='utf-8')
    return destination, archive_hash

def find_dataset_roots(extracted_roots):
    dsads_roots, har_roots = set(), set()
    for extracted in extracted_roots:
        for activity in Path(extracted).rglob('a01'):
            if activity.is_dir() and (activity / 'p1' / 's01.txt').is_file():
                dsads_roots.add(activity.parent.resolve())
        for labels in Path(extracted).rglob('y_train.txt'):
            root = labels.parent.parent
            if (root / 'test' / 'y_test.txt').is_file() and (root / 'train' / 'Inertial Signals').is_dir():
                har_roots.add(root.resolve())
    if len(dsads_roots) != 1 or len(har_roots) != 1:
        raise ValueError(f'Expected one root per dataset. Found DSADS={len(dsads_roots)}, HAR={len(har_roots)}. '
                         'Use the two official UCI ZIPs and remove duplicate copies from raw/.')
    return next(iter(dsads_roots)), next(iter(har_roots))


In [3]:
source_inventory, extracted_roots = [], []
for archive in archives:
    print('Preparing:', archive.name, flush=True)
    extracted, digest = unpack_dataset_archive(archive, LOCAL)
    extracted_roots.append(extracted)
    source_inventory.append({'filename': archive.name, 'bytes': archive.stat().st_size, 'sha256': digest})
DSADS_ROOT, HAR_ROOT = find_dataset_roots(extracted_roots)
print('Both dataset layouts were found.')


Preparing: daily+and+sports+activities.zip
Preparing: human+activity+recognition+using+smartphones.zip
Both dataset layouts were found.


## Load the windowed signals

Arrays use **[window, channel, time]** order. Labels, participant IDs and file paths are metadata,
never sensor inputs. Smartphone HAR uses total acceleration and body gyroscope only (six channels).
Derived body-acceleration channels are excluded so an accelerometer fault cannot leave an unaffected
duplicate representation in the input.


In [4]:
HAR_ACTIVITIES = ['WALKING', 'WALKING_UPSTAIRS', 'WALKING_DOWNSTAIRS', 'SITTING', 'STANDING', 'LAYING']
DSADS_ACTIVITIES = [
    'sitting', 'standing', 'lying_on_back', 'lying_on_right_side', 'ascending_stairs',
    'descending_stairs', 'standing_in_elevator', 'moving_in_elevator', 'walking_in_parking_lot',
    'treadmill_flat_4kmh', 'treadmill_incline_4kmh', 'treadmill_running_8kmh', 'stepper',
    'cross_trainer', 'cycling_horizontal', 'cycling_vertical', 'rowing', 'jumping', 'basketball'
]

def load_dsads(root):
    root = Path(root)
    pattern = re.compile(r'^a(\d{2})/p(\d+)/s(\d{2})\.txt$')
    records = []
    for path in root.glob('a[0-9][0-9]/p[0-9]*/s[0-9][0-9].txt'):
        match = pattern.fullmatch(path.relative_to(root).as_posix())
        if match:
            activity, subject, segment = map(int, match.groups())
            records.append((activity, subject, segment, path))
    records.sort(key=lambda r: r[:3])
    if not records:
        raise ValueError('No DSADS signal segments were found.')
    X = np.empty((len(records), 45, 125), dtype=np.float32)
    labels, subjects, window_ids = [], [], []
    for i, (activity, subject, segment, path) in enumerate(records):
        values = np.loadtxt(path, delimiter=',', dtype=np.float32)
        if values.shape != (125, 45):
            raise ValueError(f'Unexpected segment shape {values.shape}: {path}')
        X[i] = values.T
        labels.append(activity - 1)
        subjects.append(subject)
        window_ids.append(f'a{activity:02d}/p{subject}/s{segment:02d}')
        if (i + 1) % 2000 == 0:
            print('DSADS segments loaded:', i + 1, flush=True)
    units = ['torso', 'right_arm', 'left_arm', 'right_leg', 'left_leg']
    axes = ['acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z', 'mag_x', 'mag_y', 'mag_z']
    return {'X': X, 'y': np.asarray(labels, dtype=np.int64),
            'subjects': np.asarray(subjects, dtype=np.int16), 'window_ids': np.asarray(window_ids),
            'official_split': np.full(len(X), -1, dtype=np.int8),
            'class_names': DSADS_ACTIVITIES,
            'channel_names': [f'{unit}_{axis}' for unit in units for axis in axes],
            'sensor_groups': [{'name': unit, 'channels': list(range(i * 9, (i + 1) * 9))}
                              for i, unit in enumerate(units)]}

def load_har(root):
    root = Path(root)
    channels = ['total_acc_x', 'total_acc_y', 'total_acc_z', 'body_gyro_x', 'body_gyro_y', 'body_gyro_z']
    arrays, labels, participants, ids, official = [], [], [], [], []
    for flag, split in enumerate(['train', 'test']):
        signals = root / split / 'Inertial Signals'
        matrices = [np.loadtxt(signals / f'{channel}_{split}.txt', dtype=np.float32, ndmin=2)
                    for channel in channels]
        X = np.stack(matrices, axis=1)
        y = np.loadtxt(root / split / f'y_{split}.txt', dtype=np.int64, ndmin=1) - 1
        subjects = np.loadtxt(root / split / f'subject_{split}.txt', dtype=np.int16, ndmin=1)
        if X.shape[2] != 128 or len(X) != len(y) or len(X) != len(subjects):
            raise ValueError(f'Unexpected HAR shapes for {split}.')
        arrays.append(X)
        labels.append(y)
        participants.append(subjects)
        ids.extend(f'{split}/{i:05d}' for i in range(len(X)))
        official.extend([flag] * len(X))
    return {'X': np.concatenate(arrays), 'y': np.concatenate(labels),
            'subjects': np.concatenate(participants), 'window_ids': np.asarray(ids),
            'official_split': np.asarray(official, dtype=np.int8),
            'class_names': HAR_ACTIVITIES, 'channel_names': channels,
            'sensor_groups': [{'name': 'accelerometer', 'channels': [0, 1, 2]},
                              {'name': 'gyroscope', 'channels': [3, 4, 5]}]}

def validate_dataset(data, expected_shape, expected_subjects):
    X, y, subjects = data['X'], data['y'], data['subjects']
    assert X.shape == expected_shape, f'Expected {expected_shape}, received {X.shape}'
    assert np.isfinite(X).all(), 'Non-finite sensor values were found.'
    assert len(X) == len(y) == len(subjects) == len(data['window_ids'])
    assert len(set(data['window_ids'].tolist())) == len(X), 'Repeated window IDs.'
    assert len(np.unique(subjects)) == expected_subjects, 'Unexpected participant count.'
    assert np.array_equal(np.unique(y), np.arange(len(data['class_names']))), 'Unexpected labels.'
    group_channels = [i for group in data['sensor_groups'] for i in group['channels']]
    assert sorted(group_channels) == list(range(X.shape[1])), 'Sensor groups do not partition channels.'


In [5]:
datasets = {'dsads': load_dsads(DSADS_ROOT), 'har': load_har(HAR_ROOT)}
validate_dataset(datasets['dsads'], (9120, 45, 125), 8)
validate_dataset(datasets['har'], (10299, 6, 128), 30)
assert int((datasets['har']['official_split'] == 0).sum()) == 7352
assert int((datasets['har']['official_split'] == 1).sum()) == 2947
print('Signal arrays, class labels, participant counts and channel groups passed validation.')


DSADS segments loaded: 2000
DSADS segments loaded: 4000
DSADS segments loaded: 6000
DSADS segments loaded: 8000
Signal arrays, class labels, participant counts and channel groups passed validation.


## Create participant-disjoint splits

- **train:** fit the classifier and reconstructor; fit normalization only here.
- **policy:** generate labeled action outcomes and fit the action scorer.
- **validation:** choose thresholds and hyperparameters.
- **test:** reserved for final evaluation after all choices are frozen.

Smartphone HAR retains all nine official test participants; the 21 official training participants
are divided into 15 train, three policy and three validation participants.

DSADS has four predeclared folds. Each fold uses four train, one policy, one validation and two
test participants. Test pairs cover all eight participants once across folds. The initial pilot
uses **fold_0 validation**; the final experiment should include all four folds. Each fold requires
fresh model fitting. Model random seeds (2026, 2027, 2028) are separate from these fixed splits.


In [6]:
SPLIT_NAMES = ['train', 'policy', 'validation', 'test']

def indices_from_subjects(data, subject_groups):
    return {name: np.flatnonzero(np.isin(data['subjects'], group)).astype(np.int64)
            for name, group in subject_groups.items()}

def make_splits(name, data, seed=2026):
    subjects = np.unique(data['subjects'])
    folds = {}
    if name == 'har':
        source_subjects = np.unique(data['subjects'][data['official_split'] == 0])
        test_subjects = np.unique(data['subjects'][data['official_split'] == 1])
        assert not set(source_subjects) & set(test_subjects)
        order = np.random.default_rng(seed).permutation(source_subjects)
        subject_groups = {'train': order[:15], 'policy': order[15:18],
                          'validation': order[18:], 'test': test_subjects}
        folds['fold_0'] = indices_from_subjects(data, subject_groups)
    elif name == 'dsads':
        assert len(subjects) == 8
        for fold in range(4):
            test_subjects = subjects[2 * fold:2 * fold + 2]
            remaining = subjects[~np.isin(subjects, test_subjects)]
            order = np.random.default_rng(seed + fold).permutation(remaining)
            subject_groups = {'train': order[:4], 'policy': order[4:5],
                              'validation': order[5:6], 'test': test_subjects}
            folds[f'fold_{fold}'] = indices_from_subjects(data, subject_groups)
    else:
        raise ValueError(name)
    return folds

def audit_splits(data, folds):
    report = {}
    for fold, parts in folds.items():
        joined = np.concatenate([parts[name] for name in SPLIT_NAMES])
        assert np.array_equal(np.sort(joined), np.arange(len(data['X']))), 'Splits must partition all windows.'
        subject_sets = {name: set(data['subjects'][idx].tolist()) for name, idx in parts.items()}
        for i, first in enumerate(SPLIT_NAMES):
            assert len(parts[first]), f'Empty {first} split.'
            assert np.array_equal(np.unique(data['y'][parts[first]]),
                                  np.arange(len(data['class_names']))), f'Missing classes in {first}.'
            for second in SPLIT_NAMES[i + 1:]:
                assert not subject_sets[first] & subject_sets[second], 'Participant overlap.'
        report[fold] = {name: {'windows': len(parts[name]), 'subjects': sorted(subject_sets[name])}
                        for name in SPLIT_NAMES}
    return report

def normalization_from_training(X, train_indices):
    train = X[train_indices]
    mean = train.mean(axis=(0, 2), dtype=np.float64).astype(np.float32)
    std = train.std(axis=(0, 2), dtype=np.float64).astype(np.float32)
    return mean, np.maximum(std, np.float32(1e-6))

def window_fingerprints(X):
    return np.asarray([hashlib.sha256(np.ascontiguousarray(window).tobytes()).hexdigest() for window in X])

def cross_split_duplicate_groups(fingerprints, parts):
    owners = {}
    for split, indices in parts.items():
        for digest in np.unique(fingerprints[indices]):
            owners.setdefault(str(digest), []).append(split)
    return [{'sha256': digest, 'splits': splits} for digest, splits in owners.items() if len(splits) > 1]


In [7]:
folds_by_dataset = {name: make_splits(name, data, SEED) for name, data in datasets.items()}
subject_split_manifest = {name: audit_splits(datasets[name], folds)
                          for name, folds in folds_by_dataset.items()}
fingerprints, integrity = {}, {}
for name, data in datasets.items():
    fingerprints[name] = window_fingerprints(data['X'])
    duplicate_counts = Counter(fingerprints[name].tolist())
    conflicts = {fold: cross_split_duplicate_groups(fingerprints[name], parts)
                 for fold, parts in folds_by_dataset[name].items()}
    integrity[name] = {
        'shape': list(data['X'].shape), 'finite': True, 'participant_disjoint': True,
        'duplicate_content_groups': sum(count > 1 for count in duplicate_counts.values()),
        'cross_split_duplicate_groups': conflicts,
        'ready_for_training': not any(conflicts.values())
    }
    print(name.upper(), data['X'].shape,
          '| Ready for training:', integrity[name]['ready_for_training'])
print('HAR official test windows:', len(folds_by_dataset['har']['fold_0']['test']))


DSADS (9120, 45, 125) | Ready for training: True
HAR (10299, 6, 128) | Ready for training: True
HAR official test windows: 2947


## Save reusable inputs and the experimental protocol

The main NPZ files contain **unscaled** float32 sensor arrays plus separate labels and metadata.
Each fold has its own training-only channel means and standard deviations. Future notebooks
normalize using `X_normalized = (X - mean[None, :, None]) / std[None, :, None]`.
Faults will be generated after normalization. No corrupted test predictions are examined here.

Primary fault protocol: one affected sensor group; dropout, bias and gain during development;
stuck readings held out for final fault-transfer evaluation. A separate seen-fault evaluation
will measure performance on new recordings. Zero is a valid physical reading; explicit masking
actions must pass an availability indicator to the classifier.


In [8]:
protocol = {
    'schema_version': 1, 'project': 'SensorRepair_ICDSIS', 'split_seed': SEED,
    'array_layout': 'N,C,T', 'sensor_dtype': 'float32', 'labels_zero_based': True,
    'model_seeds': [2026, 2027, 2028], 'normalization_fit_split': 'train',
    'policy_fit_split': 'policy', 'selection_split': 'validation', 'final_evaluation_split': 'test',
    'primary_dsads_evaluation': 'four participant folds; refit models independently in every fold',
    'primary_har_evaluation': 'official test participants; fixed internal training/policy/validation split',
    'development_fault_types': ['dropout', 'bias', 'gain'], 'held_out_fault_type': 'stuck',
    'fault_location': 'one sensor group, selected without using activity labels',
    'fault_severities': {
        'dropout': {'time_fraction': [0.25, 0.5, 1.0], 'replacement': 0.0},
        'bias': {'absolute_offset_in_training_std': [0.5, 1.0, 2.0], 'sign': 'random +1 or -1 per affected channel'},
        'gain': {'factor': [1.2, 1.5, 2.0], 'direction': 'random factor or reciprocal for the affected group'},
        'stuck': {'time_fraction': [0.25, 0.5, 1.0], 'replacement': 'repeat first sample of affected interval'}
    },
    'fault_interval': 'contiguous interval with uniformly selected valid start; shared interval across group channels',
    'fault_injection_space': 'training-standardized signals',
    'fault_metadata': 'for reproducibility/evaluation only; hidden injected locations are not policy inputs',
    'datasets': {name: {'class_names': data['class_names'], 'channel_names': data['channel_names'],
                        'sensor_groups': data['sensor_groups']} for name, data in datasets.items()},
    'sources': {
        'dsads': 'https://archive.ics.uci.edu/dataset/256/daily+and+sports+activities',
        'har': 'https://archive.ics.uci.edu/dataset/240/human+activity+recognition+using+smartphones'
    },
    'license': 'CC BY 4.0; cite the respective dataset creators',
    'test_policy': 'Freeze model choices, thresholds and evaluation code before computing final test predictions.'
}

def save_npz(path, **arrays):
    with Path(path).open('wb') as stream:
        np.savez_compressed(stream, **arrays)

generated = LOCAL / 'generated'
generated.mkdir(exist_ok=True)
summary_rows = []
for name, data in datasets.items():
    arrays = {key: data[key] for key in ['X', 'y', 'subjects', 'window_ids', 'official_split']}
    arrays['window_sha256'] = fingerprints[name]
    save_npz(generated / f'{name}_signals.npz', **arrays)
    split_arrays = {}
    for fold, parts in folds_by_dataset[name].items():
        for split, indices in parts.items():
            split_arrays[f'{fold}_{split}'] = indices
            summary_rows.append({'dataset': name, 'fold': fold, 'split': split, 'windows': len(indices),
                                 'participants': len(np.unique(data['subjects'][indices])),
                                 'classes': len(np.unique(data['y'][indices])),
                                 'channels': data['X'].shape[1], 'time_steps': data['X'].shape[2]})
        mean, std = normalization_from_training(data['X'], parts['train'])
        save_npz(generated / f'{name}_{fold}_normalization.npz', mean=mean, std=std)
    save_npz(generated / f'{name}_splits.npz', **split_arrays)
    print('Prepared:', name, flush=True)

import csv
with (generated / 'dataset_summary.csv').open('w', newline='', encoding='utf-8') as stream:
    writer = csv.DictWriter(stream, fieldnames=list(summary_rows[0]))
    writer.writeheader()
    writer.writerows(summary_rows)
json_outputs = {'subject_splits.json': subject_split_manifest, 'data_integrity.json': integrity,
                'protocol.json': protocol, 'source_archives.json': source_inventory,
                'environment.json': {'python': sys.version, 'numpy': np.__version__, 'platform': platform.platform()}}
for filename, value in json_outputs.items():
    (generated / filename).write_text(json.dumps(value, indent=2), encoding='utf-8')

artifact_inventory = []
for path in sorted(generated.iterdir()):
    if path.is_file() and path.name != 'artifact_inventory.json':
        artifact_inventory.append({'filename': path.name, 'bytes': path.stat().st_size,
                                   'sha256': file_sha256(path)})
(generated / 'artifact_inventory.json').write_text(json.dumps(artifact_inventory, indent=2), encoding='utf-8')

# Publish the completion file last so interrupted writes cannot look complete.
completion = OUT / 'notebook01_complete.json'
if completion.exists():
    completion.unlink()
for path in sorted(generated.iterdir()):
    destination = OUT / path.name
    partial = destination.with_name(destination.name + '.partial')
    shutil.copyfile(path, partial)
    os.replace(partial, destination)
status = {'ready_for_training': all(record['ready_for_training'] for record in integrity.values()),
          'protocol_sha256': file_sha256(generated / 'protocol.json'),
          'artifact_inventory_sha256': file_sha256(generated / 'artifact_inventory.json'),
          'artifacts': len(artifact_inventory), 'output_folder': str(OUT)}
completion.write_text(json.dumps(status, indent=2), encoding='utf-8')
print('Saved', len(artifact_inventory), 'prepared artifacts and a completion record.')


Prepared: dsads
Prepared: har
Saved 15 prepared artifacts and a completion record.


In [9]:
print('NOTEBOOK 01 COMPLETE')
for name, data in datasets.items():
    print(f"{name.upper()}: X={data['X'].shape}, classes={len(data['class_names'])}, "
          f"participants={len(np.unique(data['subjects']))}, sensor_groups={len(data['sensor_groups'])}")
    for fold, parts in subject_split_manifest[name].items():
        print(' ', fold, '|', ' | '.join(f'{split}: {record["windows"]}' for split, record in parts.items()))
print('Participant overlap: 0 in every fold')
print('Cross-split content-duplicate groups:',
      sum(len(groups) for record in integrity.values() for groups in record['cross_split_duplicate_groups'].values()))
print('Ready for training:', status['ready_for_training'])
print('Outputs:', OUT)
if not status['ready_for_training']:
    print('Review data_integrity.json before training: identical sensor windows occur across split boundaries.')
print('Share this final output together with dataset_summary.csv and data_integrity.json.')


NOTEBOOK 01 COMPLETE
DSADS: X=(9120, 45, 125), classes=19, participants=8, sensor_groups=5
  fold_0 | train: 4560 | policy: 1140 | validation: 1140 | test: 2280
  fold_1 | train: 4560 | policy: 1140 | validation: 1140 | test: 2280
  fold_2 | train: 4560 | policy: 1140 | validation: 1140 | test: 2280
  fold_3 | train: 4560 | policy: 1140 | validation: 1140 | test: 2280
HAR: X=(10299, 6, 128), classes=6, participants=30, sensor_groups=2
  fold_0 | train: 5364 | policy: 989 | validation: 999 | test: 2947
Participant overlap: 0 in every fold
Cross-split content-duplicate groups: 0
Ready for training: True
Outputs: /content/drive/MyDrive/SensorRepair_ICDSIS/processed/v1
Share this final output together with dataset_summary.csv and data_integrity.json.


## Next notebook

Notebook 02 will train the small activity classifier and reconstructor, generate policy-training
action outcomes, and run a **validation-only** pilot comparing fixed responses with oracle action
selection. Final test predictions remain reserved until the model and evaluation choices are frozen.

Dataset references:
- Barshan, B. & Altun, K. (2010). *Daily and Sports Activities*. UCI. DOI: 10.24432/C5C59F.
- Reyes-Ortiz, J. et al. (2013). *Human Activity Recognition Using Smartphones*. UCI. DOI: 10.24432/C54S4K.
